# 🚗 Verkehrsdaten 2022-2024: Qualitätskontrolle & PDF-Reporting

Dieses Notebook verarbeitet und analysiert die historischen Verkehrsdaten (Straßensegmente mit Fahrtenzählungen) der Jahre 2022 bis 2024. Ziel ist es, die Match-Qualität älterer Daten auf das aktuelle Straßennetzwerk (2024) zu prüfen und problematische sowie besonders stark frequentierte Segmente visuell aufzubereiten.

### 📋 Ablauf der Datenverarbeitung:
1. **Datenaufbereitung:** Einlesen der Rohdaten (Shapefiles/GeoPackages) und Filterung auf Segmente mit signifikanter Auslastung (`count >= 100`).
2. **Koordinatentransformation:** Einheitliche Projektion in das metrische System UTM Zone 32N (EPSG 25832) für präzise räumliche Analysen.
3. **Räumliche Verschneidung (Spatial Joins):** 
   - Abgleich der Jahre 2022 & 2023 mit dem 2024er Referenznetz via `sjoin` (Intersects).
   - Für Segmente ohne direkten Treffer (Unmatched): Ermittlung der nächstgelegenen Referenzstraße via `sjoin_nearest` (max. 100 m Suchradius).
4. **Automatisierter PDF-Export:**
   - **Matched Prominent:** Erstellung eines Berichts mit den Top 10 der erfolgreich gematchten Segmente (zur Validierung der Zuordnung).
   - **Unmatched Top 25:** Visualisierung der 25 am stärksten befahrenen Segmente ohne direkten Treffer im aktuellen Referenznetz.

*Alle Auswertungen werden direkt als übersichtliche, kartenbasierte PDF-Berichte im persönlichen Workspace gespeichert.*

In [0]:
%pip install geopandas -q
%pip install osmnx -q
%pip install contextily -q

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Standard Library
import os
import warnings


# Third-Party: Data & Geometry
import numpy as np
import pandas as pd
from shapely.geometry import box

# Third-Party: Geospatial
import contextily as ctx
import geopandas as gpd
import osmnx as ox
from geopandas import sjoin

# Third-Party: Visualization
import matplotlib
matplotlib.use('Agg')
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# Unterdrückt spezifisch die pyogrio RuntimeWarning bzgl. 'user_version'
warnings.filterwarnings(
    "ignore", 
    message=".*unrecognized user_version.*", 
    category=RuntimeWarning
)

### Lädt die Verkehrsdaten für 2022-2024, vereinheitlicht die Spalten, filtert Segmente mit count < 100 und prüft die Spalten vor dem Zusammenführen.

In [0]:
from pathlib import Path

# Nutze pathlib für sichere und saubere Pfad-Zusammensetzungen
base_path = Path("/Volumes/traffic/raw_data/gpkg_files")

# Daten einlesen (mit dem / Operator von pathlib)
gdf_22 = gpd.read_file(base_path / "2022" / "trafficvolumes.shp")
gdf_23 = gpd.read_file(base_path / "2023" / "trafficvolumes.shp")
gdf_24 = gpd.read_file(base_path / "trafficvolumes24.gpkg")

# Jahr hinzufügen
gdf_22["year"] = 2022
gdf_23["year"] = 2023
gdf_24["year"] = 2024

# Spalte in gdf_24 anpassen
gdf_24 = gdf_24.rename(columns={"number_of_matched_trips": "count"})

# In EINEM Schritt: Segmente mit < 100 Fahrten filtern UND in das Arbeits-CRS (UTM 32N) umwandeln
gdf_22 = gdf_22[gdf_22["count"] >= 100].to_crs(epsg=25832)
gdf_23 = gdf_23[gdf_23["count"] >= 100].to_crs(epsg=25832)
gdf_24 = gdf_24[gdf_24["count"] >= 100].to_crs(epsg=25832)

# Check, ob die Spalten übereinstimmen
print(gdf_22.columns.tolist())
print(gdf_23.columns.tolist())
print(gdf_24.columns.tolist())

['count', 'geometry', 'year']
['count', 'geometry', 'year']
['osm_way_id', 'count', 'geometry', 'year']


Ordnet die Verkehrsdaten von 2022 und 2023 per Spatial Join den 2024er Referenzgeometrien mit osm_way_id zu und prüft anschließend die Match-Qualität.

In [0]:
# ─── 1. Extract reference: geometry + osm_way_id from 2024 ────────────
# Geometrien aus gdf_24 extrahieren. 
# (Hinweis: Die Daten liegen bereits im metrischen Koordinatensystem UTM Zone 32N vor)
gdf_ref = gdf_24[["osm_way_id", "geometry"]].copy()

# ─── 2. Spatial join 2022 and 2023 to 2024 reference ─────────────────
gdf_22_matched = sjoin(
    gdf_22,
    gdf_ref,
    how="left",
    predicate="intersects"
)

gdf_23_matched = sjoin(
    gdf_23,
    gdf_ref,
    how="left",
    predicate="intersects"
)

# ─── 3. Check match quality ───────────────────────────────────────────
print(f"2022 matched: {gdf_22_matched['osm_way_id'].notna().sum():,} / {len(gdf_22_matched):,}")
print(f"2023 matched: {gdf_23_matched['osm_way_id'].notna().sum():,} / {len(gdf_23_matched):,}")

2022 matched: 34,598 / 43,263
2023 matched: 45,691 / 51,089


##### Nicht gematchte Segmente — Visualisierung & PDF-Export

Filtert die Top-25-Straßensegmente ohne Treffer im Spatial Join mit dem 2024-Referenznetz (je separat für 2022 und 2023),
sortiert nach Fahrtenzahl. Für jedes Segment wird eine Kartenansicht mit OSM-Hintergrundkarte erstellt — nicht gematchte 
Segmente rot, umgebendes 2024-Referenznetz blau — und als einzelne A4-Seite in ein PDF exportiert.

In [0]:
from geopandas import sjoin_nearest

def clean_join_cols(gdf):
    return gdf.drop(columns=[c for c in gdf.columns if c in ["index_right", "index_left"]], errors="ignore")

def nearest_osm(top25_gdf, ref_gdf, max_dist=100):
    # Die Geometrien sind bereits in EPSG 25832, wir können sjoin_nearest direkt ausführen
    result = sjoin_nearest(
        clean_join_cols(top25_gdf),
        ref_gdf,
        how="left",
        max_distance=max_dist,
        distance_col="dist_to_ref_m"
    )
    # Duplikate entfernen, falls ein Segment zu mehreren Referenzlinien exakt gleich weit entfernt ist
    return result.loc[~result.index.duplicated(keep="first")]

# Nächstgelegene Segmente berechnen
gdf_22_nearest = nearest_osm(top25_22, gdf_ref)
gdf_23_nearest = nearest_osm(top25_23, gdf_ref)

# Werte in unsere Top25 DataFrames übertragen
top25_22["osm_way_id_api"] = gdf_22_nearest["osm_way_id_right"].reindex(top25_22.index).values
top25_22["dist_to_ref_m"] = gdf_22_nearest["dist_to_ref_m"].reindex(top25_22.index).values

top25_23["osm_way_id_api"] = gdf_23_nearest["osm_way_id_right"].reindex(top25_23.index).values
top25_23["dist_to_ref_m"] = gdf_23_nearest["dist_to_ref_m"].reindex(top25_23.index).values

# Segmente ohne Treffer im Umkreis verwerfen
top25_22 = top25_22.dropna(subset=["osm_way_id_api"]).reset_index(drop=True)
top25_23 = top25_23.dropna(subset=["osm_way_id_api"]).reset_index(drop=True)

print(f"2022: {len(top25_22)} Segmente mit OSM Way ID")
print(f"2023: {len(top25_23)} Segmente mit OSM Way ID")

2022: 25 Segmente mit OSM Way ID
2023: 23 Segmente mit OSM Way ID


##### Nearest-Match: OSM Way ID per räumlicher Nähe ermitteln

Für jedes nicht gematchte Segment wird über `sjoin_nearest` das nächstgelegene Segment aus dem
2024-Referenznetz (max. 100 m) gesucht und dessen OSM Way ID sowie Distanz übernommen.
Segmente ohne Treffer innerhalb des Suchradius werden verworfen.


In [0]:
from geopandas import sjoin_nearest # <-- Hier ist der Import

def clean_join_cols(gdf):
    return gdf.drop(columns=[c for c in gdf.columns if c in ["index_right", "index_left"]], errors="ignore")

def nearest_osm(top25_gdf, ref_gdf, max_dist=100):
    result = sjoin_nearest(
        clean_join_cols(top25_gdf), # <-- HIER: Das .to_crs() ist jetzt weg!
        ref_gdf,
        how="left",
        max_distance=max_dist,
        distance_col="dist_to_ref_m"
    )
    return result.loc[~result.index.duplicated(keep="first")]

gdf_22_nearest = nearest_osm(top25_22, gdf_ref)
gdf_23_nearest = nearest_osm(top25_23, gdf_ref)

top25_22["osm_way_id_api"] = gdf_22_nearest["osm_way_id_right"].reindex(top25_22.index).values
top25_22["dist_to_ref_m"]  = gdf_22_nearest["dist_to_ref_m"].reindex(top25_22.index).values

top25_23["osm_way_id_api"] = gdf_23_nearest["osm_way_id_right"].reindex(top25_23.index).values
top25_23["dist_to_ref_m"]  = gdf_23_nearest["dist_to_ref_m"].reindex(top25_23.index).values

# Drop NAs
top25_22 = top25_22.dropna(subset=["osm_way_id_api"]).reset_index(drop=True)
top25_23 = top25_23.dropna(subset=["osm_way_id_api"]).reset_index(drop=True)

print(f"2022: {len(top25_22)} Segmente mit OSM Way ID")
print(f"2023: {len(top25_23)} Segmente mit OSM Way ID")

2022: 25 Segmente mit OSM Way ID
2023: 23 Segmente mit OSM Way ID


##### Gematchte Segmente — Diverse Auswahl Top 10

Wählt die 10 Segmente mit der höchsten Fahrtenzahl aus dem erfolgreich gematchten Anteil aus,
dabei wird pro OSM Way ID nur ein Segment berücksichtigt, um Cluster auf derselben Straße zu vermeiden.


In [0]:
# ─── 5. Prominent matched segments — diverse selection ────────────────
# Pick top matched segments, then deduplicate by osm_way_id to avoid clustering

def select_diverse_matched(gdf_matched, n=10):
    """Top matched segments, one per OSM Way ID to ensure diversity."""
    matched = gdf_matched[gdf_matched["osm_way_id"].notna()].copy()
    matched = matched.sort_values("count", ascending=False)
    # One segment per unique OSM way
    diverse = matched.drop_duplicates(subset="osm_way_id", keep="first").head(n)
    return diverse.reset_index(drop=True)

matched_22 = select_diverse_matched(gdf_22_matched, n=10)
matched_23 = select_diverse_matched(gdf_23_matched, n=10)

print(f"2022 diverse matched: {len(matched_22)} Segmente")
print(matched_22[["osm_way_id", "count"]].to_string())
print(f"\n2023 diverse matched: {len(matched_23)} Segmente")
print(matched_23[["osm_way_id", "count"]].to_string())


2022 diverse matched: 10 Segmente
     osm_way_id  count
0  1.218174e+08   2931
1  4.588479e+08   2890
2  2.448196e+07   2890
3  2.336551e+08   2704
4  1.529293e+08   2693
5  1.834134e+08   2676
6  1.033603e+08   2660
7  6.945540e+08   2660
8  2.726051e+07   2660
9  1.180777e+09   2660

2023 diverse matched: 10 Segmente
     osm_way_id  count
0  1.218174e+08   2689
1  1.529293e+08   2672
2  6.945540e+08   2656
3  1.033603e+08   2656
4  1.180777e+09   2656
5  2.726051e+07   2656
6  8.424561e+08   2564
7  8.424561e+08   2564
8  8.424561e+08   2564
9  1.333024e+08   2539


##### Plot-Funktionen & PDF-Builder

Definiert die Hilfsfunktionen für die Kartendarstellung und den PDF-Export. `plot_matched_page`
zeigt ein erfolgreich gematchtes Segment (blau) zusammen mit dem entsprechenden 2024-Referenzsegment
(grün), `plot_unmatched_page` stellt nicht gematchte Segmente (rot) mit dem umliegenden Referenznetz
(blau) dar — jeweils auf einer A4-Querseite mit CartoDB-Hintergrundkarte. `build_pdf` iteriert über
alle Segmente und schreibt jede Karte als einzelne Seite in ein PDF.


In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.backends.backend_pdf import PdfPages
import contextily as ctx
from shapely.geometry import box as sbox

def plot_matched_page(ax, seg_row, ref_gdf, rank, year):
    """
    Show original segment (blue) and matched 2024 reference segment (green).
    """
    seg_3857 = seg_row.to_crs(epsg=3857)
    osm_id   = seg_row["osm_way_id"].iloc[0]
    count_val = int(seg_row["count"].iloc[0])

    # Find the matched 2024 reference segment (ref_gdf ist bereits in EPSG:3857)
    ref_match = ref_gdf[ref_gdf["osm_way_id"] == osm_id]

    # Map extent
    BUFFER_M = 500
    all_geoms = pd.concat([seg_3857, ref_match]) if not ref_match.empty else seg_3857
    minx, miny, maxx, maxy = all_geoms.total_bounds
    cx, cy = (minx + maxx) / 2, (miny + maxy) / 2
    half = max((maxx - minx), (maxy - miny)) / 2 + BUFFER_M

    # Plot
    if not ref_match.empty:
        ref_match.plot(ax=ax, color="#4CAF50", linewidth=6, alpha=0.5, zorder=2, label="2024 Referenz")
    seg_3857.plot(ax=ax, color="#1565C0", linewidth=3, zorder=3, label=f"Original {year}")

    ax.set_xlim(cx - half, cx + half)
    ax.set_ylim(cy - half, cy + half)
    ax.set_aspect("equal")
    ax.set_axis_off()

    ctx.add_basemap(ax, crs="EPSG:3857", source=ctx.providers.CartoDB.Positron,
                    zoom="auto", attribution=False, zorder=1)

    ax.set_title(
        f"Rang #{rank}  |  Jahr {year}  |  OSM Way ID: {int(osm_id)}  |  Fahrten: {count_val:,}\n"
        f"Matched — Original (blau) vs. 2024 Referenz (grün)",
        fontsize=10, fontweight="bold", pad=8, loc="left"
    )

    handles = [
        mpatches.Patch(color="#1565C0", label=f"Original {year}"),
        mpatches.Patch(color="#4CAF50", alpha=0.5, label="2024 Referenz"),
    ]
    ax.legend(handles=handles, loc="lower right", fontsize=8, framealpha=0.9)


def plot_unmatched_page(ax, seg_row, ref_gdf, rank, year):
    """
    Show unmatched segment (red) with nearby 2024 reference (grey).
    """
    seg_3857 = seg_row.to_crs(epsg=3857)
    ref_3857 = ref_gdf  # ref_gdf ist bereits in EPSG:3857
    geom     = seg_3857.geometry.iloc[0]
    count_val = int(seg_row["count"].iloc[0])
    osm_id    = seg_row["osm_way_id_api"].iloc[0] if pd.notna(seg_row["osm_way_id_api"].iloc[0]) else "–"
    dist      = seg_row["dist_to_ref_m"].iloc[0] if "dist_to_ref_m" in seg_row.columns else None

    BUFFER_M = 500
    minx, miny, maxx, maxy = geom.bounds
    cx, cy = (minx + maxx) / 2, (miny + maxy) / 2
    half = max((maxx - minx), (maxy - miny)) / 2 + BUFFER_M

    bbox = sbox(cx - half, cy - half, cx + half, cy + half)
    ref_clip = ref_3857[ref_3857.geometry.intersects(bbox)]

    if not ref_clip.empty:
        ref_clip.plot(ax=ax, color="#2196F3", linewidth=1.2, alpha=0.6, zorder=2)
    seg_3857.plot(ax=ax, color="#e63946", linewidth=5, zorder=3)

    ax.set_xlim(cx - half, cx + half)
    ax.set_ylim(cy - half, cy + half)
    ax.set_aspect("equal")
    ax.set_axis_off()

    ctx.add_basemap(ax, crs="EPSG:3857", source=ctx.providers.CartoDB.Positron,
                    zoom="auto", attribution=False, zorder=1)

    dist_str = f"  |  Distanz: {dist:.1f} m" if dist and pd.notna(dist) else ""
    ax.set_title(
        f"Rang #{rank}  |  Jahr {year}  |  Nächste OSM Way ID: {int(osm_id) if osm_id != '–' else '–'}  |  "
        f"Fahrten: {count_val:,}{dist_str}\nNicht gematchtes Segment",
        fontsize=10, fontweight="bold", pad=8, loc="left"
    )

    handles = [
        mpatches.Patch(color="#e63946", label="Nicht gematchtes Segment"),
        mpatches.Patch(color="#2196F3", alpha=0.6, label="Referenznetz 2024"),
    ]
    ax.legend(handles=handles, loc="lower right", fontsize=8, framealpha=0.9)


def build_pdf(gdf_list, plot_fn, ref_gdf, out_path):
    """
    gdf_list : list of (gdf, year) tuples
    plot_fn  : plot_matched_page or plot_unmatched_page
    """
    # Einmalig ins Web-Mercator-CRS umwandeln — nicht bei jeder Seite neu
    print("Projiziere Referenznetz für den PDF-Export nach EPSG:3857...")
    ref_gdf_3857 = ref_gdf.to_crs(epsg=3857)

    with PdfPages(out_path) as pdf:
        for gdf, year in gdf_list:
            for i in range(len(gdf)):
                seg = gdf.iloc[[i]]
                fig, ax = plt.subplots(figsize=(11.69, 8.27))  # A4 landscape
                fig.subplots_adjust(left=0.03, right=0.97, top=0.90, bottom=0.05)

                # Aufruf der Plot-Funktion mit dem vor-projizierten Referenznetz
                plot_fn(ax, seg, ref_gdf_3857, rank=i + 1, year=year)

                fig.text(0.5, 0.01, f"Seite {i + 1} / {len(gdf)}  |  {year}",
                         ha="center", fontsize=7, color="#888888")

                pdf.savefig(fig, dpi=150)
                plt.close(fig)
                print(f"  {year} — Seite {i + 1}/{len(gdf)}")

    print(f"✓ PDF erfolgreich gespeichert: {out_path}")

##### PDF-Export: Gematchte Segmente

Exportiert die Top-10-Segmente mit erfolgreichem Match (2022 & 2023) in ein gemeinsames PDF
und speichert es direkt im Databricks Workspace des aktuellen Nutzers.


In [0]:
from pathlib import Path

username = spark.sql("SELECT current_user()").collect()[0][0]
out_dir = Path(f"/Workspace/Users/{username}")
os.makedirs(out_dir, exist_ok=True)

print("Erstelle PDF: Prominent matched segments ...")
build_pdf(
    [(matched_22, 2022), (matched_23, 2023)],
    plot_matched_page,
    gdf_ref,
    out_dir / "matched_prominent.pdf"
)

Erstelle PDF: Prominent matched segments ...
Projiziere Referenznetz für den PDF-Export nach EPSG:3857...
  2022 — Seite 1/10
  2022 — Seite 2/10
  2022 — Seite 3/10
  2022 — Seite 4/10
  2022 — Seite 5/10
  2022 — Seite 6/10
  2022 — Seite 7/10
  2022 — Seite 8/10
  2022 — Seite 9/10
  2022 — Seite 10/10
  2023 — Seite 1/10
  2023 — Seite 2/10
  2023 — Seite 3/10
  2023 — Seite 4/10
  2023 — Seite 5/10
  2023 — Seite 6/10
  2023 — Seite 7/10
  2023 — Seite 8/10
  2023 — Seite 9/10
  2023 — Seite 10/10
✓ PDF erfolgreich gespeichert: /Workspace/Users/arne.warnke@gmail.com/matched_prominent.pdf


In [0]:
print("Erstelle PDF: Unmatched top segments ...")
build_pdf(
    [(top25_22, 2022), (top25_23, 2023)],
    plot_unmatched_page,
    gdf_ref,
    out_dir + "unmatched_top25.pdf"
)


Erstelle PDF: Unmatched top segments ...
  2022 — Seite 1/25
  2022 — Seite 2/25
  2022 — Seite 3/25
  2022 — Seite 4/25
  2022 — Seite 5/25
  2022 — Seite 6/25
  2022 — Seite 7/25
  2022 — Seite 8/25
  2022 — Seite 9/25
  2022 — Seite 10/25
  2022 — Seite 11/25
  2022 — Seite 12/25
  2022 — Seite 13/25
  2022 — Seite 14/25
  2022 — Seite 15/25
  2022 — Seite 16/25
  2022 — Seite 17/25
  2022 — Seite 18/25
  2022 — Seite 19/25
  2022 — Seite 20/25
  2022 — Seite 21/25
  2022 — Seite 22/25
  2022 — Seite 23/25
  2022 — Seite 24/25
  2022 — Seite 25/25
  2023 — Seite 1/23
  2023 — Seite 2/23
  2023 — Seite 3/23
  2023 — Seite 4/23
  2023 — Seite 5/23
  2023 — Seite 6/23
  2023 — Seite 7/23
  2023 — Seite 8/23
  2023 — Seite 9/23
  2023 — Seite 10/23
  2023 — Seite 11/23
  2023 — Seite 12/23
  2023 — Seite 13/23
  2023 — Seite 14/23
  2023 — Seite 15/23
  2023 — Seite 16/23
  2023 — Seite 17/23
  2023 — Seite 18/23
  2023 — Seite 19/23
  2023 — Seite 20/23
  2023 — Seite 21/23
  2023 — Se